In [ ]:
#| default_exp machine_learning.semantic_search

In [ ]:
#| export
import os
import re
import time
import hashlib
import fnmatch
import pathlib
from typing import List, Union, Optional, Dict, Any, Iterable, Callable, Type

import weaviate
from weaviate.classes.config import Configure, DataType, Property, VectorDistances
from weaviate.classes.query import Filter
from weaviate.util import generate_uuid5

from fastcore.basics import patch
from tqdm import tqdm


In [ ]:
import os
import random
import unittest.mock as mock
from pathlib import Path

from fastcore.test import *

In [ ]:
#| export

def latex_comment_stripping_processor(path: Union[str, os.PathLike]) -> str:
    r"""
    Opens a file and removes LaTeX comments while ignoring escaped percents (\%).
    Targets lines starting with % (not preceded by \) until the end of the line.
    """
    try:
        with open(os.fspath(path), "r", encoding="utf-8") as f:
            text = f.read()
        
        # Regex Breakdown:
        # (?<!\\) -> Lookbehind: Ensure the previous character is NOT a backslash
        # %.      -> Match the % and everything following it on that line
        pattern = r"(?<!\\)%.*"
        
        # Remove comments and strip trailing whitespace from affected lines
        cleaned_text = re.sub(pattern, "", text)
        return cleaned_text
        
    except Exception as e:
        print(f"Error reading {path}: {e}")
        return ""

In [ ]:
#| export

PathType = Union[str, os.PathLike]
FileProcessor = Callable[[PathType], str]

class MathBrainClient:
    def __init__(
        self, 
        host: str = "localhost", 
        port: int = 8080,
        chunk_size: int = 2000, 
        overlap: int = 200,
        # batch_size: int = 100    
    ) -> None:
        self.client = weaviate.connect_to_local(host=host, port=port)
        self.CHUNK_SIZE = chunk_size
        self.OVERLAP = overlap
        # self.BATCH_SIZE = batch_size


In [ ]:
#| export
@patch
def setup_collection(
        self: MathBrainClient,
        collection_name: str,
        force_recycle: bool = False
        ) -> None:
    track_name = f"{collection_name}_tracking"
    
    if force_recycle:
        for name in [collection_name, track_name]:
            if self.client.collections.exists(name):
                print(f"Force Recycle: Deleting {name}...")
                self.client.collections.delete(name)

    # Main Collection
    if not self.client.collections.exists(collection_name):
        self.client.collections.create(
            name=collection_name,
            vectorizer_config=Configure.Vectorizer.text2vec_ollama(
                api_endpoint="http://ollama:11434",
                model="nomic-embed-text",
            ),
            vector_index_config=Configure.VectorIndex.hnsw(distance_metric=VectorDistances.COSINE),
            properties=[
                Property(name="content", data_type=DataType.TEXT),
                Property(name="fileName", data_type=DataType.TEXT, skip_vectorization=True),
                Property(name="filePath", data_type=DataType.TEXT, skip_vectorization=True),
                Property(name="contentHash", data_type=DataType.TEXT, skip_vectorization=True),
            ]
        )
    
    # Tracking Collection (No vectors, just metadata for resume-logic)
    if not self.client.collections.exists(track_name):
        self.client.collections.create(
            name=track_name,
            vectorizer_config=None, 
            properties=[
                Property(name="filePath", data_type=DataType.TEXT),
                Property(name="contentHash", data_type=DataType.TEXT),
                Property(name="status", data_type=DataType.TEXT),
            ]
        )
    print(f"Collections initialized: {collection_name}")

In [ ]:
#| export
@patch
def _split_text(
        self: MathBrainClient,
        text: str
        ) -> List[str]:
    """LaTeX-aware chunking."""
    env_pattern = r'(\\begin\{.*?\}.*?\\end\{.*?\})'
    parts = re.split(env_pattern, text, flags=re.DOTALL)
    chunks, current_chunk = [], ""

    for part in parts:
        part = part.strip()
        if not part: continue
        if len(current_chunk) + len(part) > self.CHUNK_SIZE:
            if current_chunk: chunks.append(current_chunk.strip())
            if len(part) > self.CHUNK_SIZE:
                step = self.CHUNK_SIZE - self.OVERLAP
                for i in range(0, len(part), step):
                    chunks.append(part[i : i + self.CHUNK_SIZE])
                current_chunk = ""
            else:
                overlap_text = current_chunk[-self.OVERLAP:] if len(current_chunk) > self.OVERLAP else ""
                current_chunk = overlap_text + part + "\n\n"
        else:
            current_chunk += part + "\n\n"
    if current_chunk: chunks.append(current_chunk.strip())
    return [c for c in chunks if len(c) > 20]


In [ ]:
#| export
@patch
def _get_file_hash(
        self: MathBrainClient,
        text: str
        ) -> str:
    return hashlib.md5(text.encode('utf-8')).hexdigest()


In [ ]:
#| hide


# Setup a dummy client for logic tests (avoiding real DB connection here)
dummy_client = MathBrainClient.__new__(MathBrainClient)
dummy_client.CHUNK_SIZE = 100
dummy_client.OVERLAP = 20

# Test: Hash Consistency
h1 = dummy_client._get_file_hash("Hello LaTeX $E=mc^2$")
h2 = dummy_client._get_file_hash("Hello LaTeX $E=mc^2$")
test_eq(h1, h2)
test_ne(h1, dummy_client._get_file_hash("different text"))

# Test: LaTeX-aware splitting
latex_text = "Intro text. \\begin{equation} x=1 \\end{equation} Outro text."
chunks = dummy_client._split_text(latex_text)
# Ensure the environment stayed intact
test_eq(any("\\begin{equation}" in c and "\\end{equation}" in c for c in chunks), True)

# Test: Chunk size filtering (should drop chunks < 20 chars per your code)
tiny_text = "a" * 10 
test_eq(len(dummy_client._split_text(tiny_text)), 0)

In [ ]:
#| export
@patch
def _get_all_paths(
    self: MathBrainClient, 
    input_source: Union[PathType, Iterable[PathType]],
    ignores: List[str]
) -> List[str]:
    """Gather and filter file paths based on extensions and ignore patterns."""
    if not isinstance(input_source, (str, os.PathLike)) or not os.path.isdir(input_source):
        return [os.path.normpath(os.fspath(p)) for p in input_source]
    
    paths = []
    for root, _, files in os.walk(input_source):
        for f in files:
            full_p = os.path.normpath(os.path.join(root, f))
            if f.lower().endswith((".tex", ".md", ".txt")):
                if not any(fnmatch.fnmatch(f, pat) or fnmatch.fnmatch(full_p, pat) for pat in ignores):
                    paths.append(full_p)
    return paths

In [ ]:
#| export
@patch
def _get_completed_files(
    self: MathBrainClient, 
    track_coll: weaviate.collections.Collection
) -> Dict[str, str]:
    """Load completed file paths and their hashes from the tracking collection."""
    print(f"Checking tracking database for resume point...")
    completed = {}
    for obj in track_coll.iterator(return_properties=["filePath", "contentHash", "status"]):
        if obj.properties.get("status") == "COMPLETED":
            completed[obj.properties["filePath"]] = obj.properties["contentHash"]
    return completed

In [ ]:
#| export
# @patch
# def _process_single_file(
#     self: MathBrainClient,
#     path: PathType,
#     processor: Callable,
#     completed_files: Dict[str, str],
#     main_coll: weaviate.collections.Collection,
#     track_coll: weaviate.collections.Collection,
#     batch: Any
# ) -> None:
#     """Hash, chunk, and upload. Manual flush used for heavy file stability."""
#     path_str, text = str(path), processor(path)
#     if not text.strip(): return
#     curr_hash = self._get_file_hash(text)

#     if path_str in completed_files and completed_files[path_str] == curr_hash: return

#     for coll in [main_coll, track_coll]:
#         coll.data.delete_many(where=Filter.by_property("filePath").equal(path_str))

#     chunks = self._split_text(text)
#     for i, chunk in enumerate(chunks):
#         batch.add_object(
#             properties={"content": chunk, "fileName": os.path.basename(path_str), 
#                         "filePath": path_str, "contentHash": curr_hash},
#             uuid=generate_uuid5(f"{path_str}_{i}"))
    
#     # If using fixed batching, a flush here ensures the file is 'sent' 
#     # before we write to the tracking collection.
#     if hasattr(batch, 'flush'): batch.flush()

#     track_coll.data.insert(
#         properties={"filePath": path_str, "contentHash": curr_hash, "status": "COMPLETED"},
#         uuid=generate_uuid5(f"track_{path_str}"))

#| export

#| export
@patch
def _process_single_file(
    self: MathBrainClient,
    path: PathType,
    processor: Callable,
    completed_files: Dict[str, str],
    main_coll: Any,
    track_coll: Any,
    batch: Any,
    batch_size: Optional[int] = None,
    pos: int = 1
) -> None:
    """Process a file with a progress bar that tracks actual upload progress."""
    path_str, text = str(path), processor(path)
    if not text.strip(): return
    curr_hash = self._get_file_hash(text)
    fname = os.path.basename(path_str) # Define fname early

    if path_str in completed_files and completed_files[path_str] == curr_hash: return

    for coll in [main_coll, track_coll]:
        coll.data.delete_many(where=Filter.by_property("filePath").equal(path_str))

    chunks = self._split_text(text)
    chunk_pbar = tqdm(chunks, desc=f"  └ {fname[:15]}", position=pos, leave=True)
    
    for i, chunk in enumerate(chunk_pbar):
        batch.add_object(
            properties={"content": chunk, "fileName": fname, 
                        "filePath": path_str, "contentHash": curr_hash},
            uuid=generate_uuid5(f"{path_str}_{i}"))
        
        if batch_size and hasattr(batch, 'flush') and (i + 1) % batch_size == 0:
            chunk_pbar.set_postfix_str("Embedding...")
            batch.flush()
    
    if hasattr(batch, 'flush'): batch.flush()
        
    track_coll.data.insert(
        properties={"filePath": path_str, "contentHash": curr_hash, "status": "COMPLETED"},
        uuid=generate_uuid5(f"track_{path_str}"))
    
    chunk_pbar.close()

In [ ]:
#| export
# @patch
# def ingest_files(
#     self: MathBrainClient, 
#     input_source: Union[PathType, Iterable[PathType]], 
#     collection_name: str = "MathDocument", 
#     force_recycle: bool = False, 
#     processor: Optional[FileProcessor] = None,
#     exclude_patterns: Optional[List[str]] = None
# ) -> None:
#     """Ingest files into Weaviate with resume capability and LaTeX-aware chunking."""
#     self.setup_collection(collection_name, force_recycle)
#     main_coll, track_coll = self.client.collections.get(collection_name), self.client.collections.get(f"{collection_name}_tracking")
    
#     proc = processor or (lambda p: open(p, "r", encoding="utf-8").read())
#     all_paths = self._get_all_paths(input_source, exclude_patterns or [])
#     completed = self._get_completed_files(track_coll)
    
#     pbar = tqdm(all_paths, desc="MathBrain Sync")
#     with main_coll.batch.dynamic() as batch:
#         for path in pbar:
#             pbar.set_postfix({"file": os.path.basename(str(path))[:20]})
#             try: self._process_single_file(path, proc, completed, main_coll, track_coll, batch)
#             except Exception as e: print(f"\n[Error] {path}: {e}")

#     final_count = main_coll.aggregate.over_all(total_count=True).total_count
#     print(f"\n✅ Sync Complete. Collection total: {final_count} objects.")

In [ ]:
#| export
# @patch
# def ingest_files(
#     self: MathBrainClient, 
#     input_source: Union[PathType, Iterable[PathType]], 
#     collection_name: str = "MathDocument", 
#     force_recycle: bool = False, 
#     processor: Optional[FileProcessor] = None,
#     exclude_patterns: Optional[List[str]] = None,
#     batch_size: Optional[int] = None # Overrides dynamic if set
# ) -> None:
#     """Ingest files with configurable batching: dynamic or fixed-size."""
#     self.setup_collection(collection_name, force_recycle)
#     main_coll = self.client.collections.get(collection_name)
#     track_coll = self.client.collections.get(f"{collection_name}_tracking")
    
#     proc = processor or (lambda p: open(p, "r", encoding="utf-8").read())
#     paths = self._get_all_paths(input_source, exclude_patterns or [])
#     completed = self._get_completed_files(track_coll)
    
#     # Choose between dynamic auto-scaling or fixed-size batching
#     batch_mgr = main_coll.batch.dynamic() if batch_size is None else \
#                 main_coll.batch.fixed_size(batch_size=batch_size)
    
#     with batch_mgr as batch:
#         pbar = tqdm(paths, desc="MathBrain Sync")
#         for p in pbar:
#             pbar.set_postfix({"file": os.path.basename(str(p))[:20]})
#             try: 
#                 self._process_single_file(p, proc, completed, main_coll, track_coll, batch)
#             except Exception as e: print(f"\n[Error] {p}: {e}")

#     count = main_coll.aggregate.over_all(total_count=True).total_count
#     print(f"\n✅ Sync Complete. Total: {count} objects.")

In [ ]:
#| export
@patch
def ingest_files(
    self: MathBrainClient, 
    input_source: Union[PathType, Iterable[PathType]], 
    collection_name: str = "MathDocument", 
    force_recycle: bool = False, 
    processor: Optional[FileProcessor] = None,
    exclude_patterns: Optional[List[str]] = None,
    batch_size: Optional[int] = None
) -> None:
    """Ingest files with nested progress bars for file and chunk tracking."""
    self.setup_collection(collection_name, force_recycle)
    main_coll = self.client.collections.get(collection_name)
    track_coll = self.client.collections.get(f"{collection_name}_tracking")
    
    proc = processor or (lambda p: open(p, "r", encoding="utf-8").read())
    paths = self._get_all_paths(input_source, exclude_patterns or [])
    completed = self._get_completed_files(track_coll)
    
    batch_mgr = main_coll.batch.dynamic() if batch_size is None else \
                main_coll.batch.fixed_size(batch_size=batch_size)
    
    with batch_mgr as batch:
        # Position 0 is the top bar (Files)
        pbar = tqdm(paths, desc="Files", position=0)
        for p in pbar:
            fname = os.path.basename(str(p))
            pbar.set_postfix({"current": fname[:20]})
            try: 
                # Pass position 1 to create the sub-bar
                self._process_single_file(p, proc, completed, main_coll, track_coll, batch, pos=1)
            except Exception as e: print(f"\n[Error] {p}: {e}")

    print(f"\n✅ Sync Complete. Total: {main_coll.aggregate.over_all(total_count=True).total_count} objects.")

In [ ]:
#| export
@patch
def close(self: MathBrainClient): self.client.close()

@patch
def __enter__(self: MathBrainClient): return self

@patch
def __exit__(
    self: MathBrainClient,
    *args): self.close()

@patch
def delete_collection(
        self: MathBrainClient, collection_name: str
        ):
    for name in [collection_name, f"{collection_name}_tracking"]:
        if self.client.collections.exists(name):
            self.client.collections.delete(name)
    print(f"Collection and Tracking deleted.")

In [ ]:
#| hide

# Ensure we mock the connection so __init__ doesn't fail
with mock.patch('weaviate.connect_to_local') as mock_connect:
    mock_instance = mock_connect.return_value
    
    # Initialize the client
    client = MathBrainClient(host="test", port=123)
    
    # Test the context manager protocol manually if 'with' still trips up
    with client as c:
        test_eq(c.client, mock_instance)
    
    # Verify close was called on exit
    test_eq(mock_instance.close.called, True)

In [ ]:
#| hide


mock_walk_data = [
    (os.path.normpath('/data'), ['subdir'], ['test.tex', 'ignore.exe', 'notes.md']),
    (os.path.normpath('/data/subdir'), [], ['hidden.txt'])
]

# We must mock os.path.isdir so the code doesn't skip to the 'else' block
with mock.patch('os.walk') as mock_walk, \
     mock.patch('os.path.isdir') as mock_isdir:
    
    mock_walk.return_value = mock_walk_data
    mock_isdir.side_effect = lambda p: p == os.path.normpath('/data')
    
    with mock.patch.object(MathBrainClient, 'setup_collection'):
        client = MathBrainClient.__new__(MathBrainClient)
        
        # Setup Mocks
        mock_weaviate = mock.MagicMock() 
        client.client = mock_weaviate
        mock_main_coll = mock.MagicMock()
        mock_track_coll = mock.MagicMock()
        
        client.client.collections.get.side_effect = lambda name: (
            mock_track_coll if "tracking" in name else mock_main_coll
        )
        
        mock_batch = mock_main_coll.batch.dynamic.return_value
        mock_batch.__enter__.return_value = mock_batch 
        mock_track_coll.iterator.return_value = []
        mock_main_coll.aggregate.over_all.return_value.total_count = 2

        processed_files = [] 
        def mock_proc(p): 
            processed_files.append(str(p))
            return "This content is definitely longer than twenty characters."
        
        client.CHUNK_SIZE = 2000
        client.OVERLAP = 200
        client._get_file_hash = lambda x: "hash123"
        client._split_text = lambda x: ["Valid chunk content"]

        # Execute - using the same path we mocked as a directory
        test_path = os.path.normpath('/data')
        client.ingest_files(test_path, processor=mock_proc, exclude_patterns=['*hidden*'])
        
        print(f"DEBUG: Files processed: {processed_files}")
        
        # Now it should be 2: test.tex and notes.md
        test_eq(len(processed_files), 2) 
        
        normalized_processed = [str(Path(p)) for p in processed_files]
        test_eq(any('test.tex' in p for p in normalized_processed), True)
        test_eq(any('notes.md' in p for p in normalized_processed), True)

Checking tracking database for resume point...


Files: 100%|██████████| 2/2 [00:00<00:00, 268.19it/s, current=notes.md]


✅ Sync Complete. Total: 2 objects.
DEBUG: Files processed: ['\\data\\test.tex', '\\data\\notes.md']


In [ ]:
#| hide
import unittest.mock as mock
from fastcore.test import *

# Setup a clean mock environment
with mock.patch('weaviate.connect_to_local'):
    client = MathBrainClient.__new__(MathBrainClient)
    client.client = mock.MagicMock()
    
    mock_main = mock.MagicMock()
    mock_track = mock.MagicMock()
    client.client.collections.get.side_effect = lambda n: mock_track if "track" in n else mock_main

    # --- Test 1: Orchestrator ---
    with mock.patch.object(client, '_get_all_paths', return_value=["test.tex"]), \
         mock.patch.object(client, '_get_completed_files', return_value={}), \
         mock.patch.object(client, 'setup_collection'), \
         mock.patch.object(client, '_process_single_file'):
        
        client.ingest_files(input_source=["test.tex"], batch_size=42, processor=mock.Mock())
        mock_main.batch.fixed_size.assert_called_with(batch_size=42)

    # --- Test 2: Verify Flushing Logic (Fixed Batch) ---
    mock_fixed_batch = mock.MagicMock()
    mock_proc = lambda p: "Dummy text for chunking."
    client._get_file_hash = lambda x: "hash123"
    client._split_text = lambda x: ["chunk1", "chunk2"]
    
    # We pass batch_size=1 to trigger the flush logic inside the loop
    client._process_single_file(
        path="test.tex",
        processor=mock_proc,
        completed_files={},
        main_coll=mock_main,
        track_coll=mock_track,
        batch=mock_fixed_batch,
        batch_size=1 
    )
    
    test_eq(mock_fixed_batch.flush.called, True)
    test_eq(mock_fixed_batch.add_object.called, True)
    
    # --- Test 3: Dynamic Batch (No Flush) ---
    mock_dynamic_batch = mock.MagicMock()
    del mock_dynamic_batch.flush 
    
    client._process_single_file(
        path="test2.tex",
        processor=mock_proc,
        completed_files={},
        main_coll=mock_main,
        track_coll=mock_track,
        batch=mock_dynamic_batch,
        batch_size=None # Dynamic mode
    )
    
    test_eq(mock_dynamic_batch.add_object.called, True)
    print("Success: Processed correctly without BATCH_SIZE attribute.")

Files: 100%|██████████| 1/1 [00:00<?, ?it/s, current=test.tex]



✅ Sync Complete. Total: <MagicMock name='mock.aggregate.over_all().total_count' id='1935238682800'> objects.


  └ test2.tex: 100%|██████████| 2/2 [00:00<?, ?it/s]

Success: Processed correctly without BATCH_SIZE attribute.


In [ ]:
#| hide
import unittest.mock as mock
from fastcore.test import *
import __main__ 

# 1. Setup Mock Environment
with mock.patch('weaviate.connect_to_local'):
    client = MathBrainClient.__new__(MathBrainClient)
    client.client = mock.MagicMock()
    mock_main = mock.MagicMock()
    mock_track = mock.MagicMock()
    client.client.collections.get.side_effect = lambda n: mock_track if "track" in n else mock_main

    # 2. Patch tqdm in the global namespace
    from tqdm.auto import tqdm as real_tqdm
    with mock.patch('__main__.tqdm', wraps=real_tqdm) as mock_tqdm:
        
        # Setup dummy data
        test_paths = ["file1.tex", "file2.tex"]
        client._get_all_paths = mock.Mock(return_value=test_paths)
        client._get_completed_files = mock.Mock(return_value={})
        client.setup_collection = mock.Mock()
        
        # FIXED: Added batch_size to the signature
        def fake_proc_file(path, proc, completed, main, track, batch, batch_size=None, pos=1):
            # This triggers the 'mock_tqdm' and should now succeed
            with tqdm(["chunk1", "chunk2"], desc="inner", position=pos, leave=True) as pbar:
                pass
            
        # Temporarily swap the real method for our fake spy method
        with mock.patch.object(client, '_process_single_file', side_effect=fake_proc_file):
            
            # 3. Execute the ingest
            client.ingest_files(input_source=test_paths, batch_size=2)

            # 4. Assertions
            # The first call is the outer 'Files' bar
            test_eq(len(mock_tqdm.call_args_list) > 1, True) # Ensure we actually have nested calls
            
            main_bar_call = mock_tqdm.call_args_list[0]
            test_eq(main_bar_call.kwargs.get('position', 0), 0)
            
            # The second call (index 1) is the inner 'chunks' bar from the first file
            inner_bar_call = mock_tqdm.call_args_list[1]
            test_eq(inner_bar_call.kwargs.get('position'), 1)
            
            print(f"Verified: tqdm called {mock_tqdm.call_count} times.")
            print("Success: Nested bar structure confirmed with matching arguments.")

Files:   0%|          | 0/2 [00:00<?, ?it/s]

inner:   0%|          | 0/2 [00:00<?, ?it/s]

inner:   0%|          | 0/2 [00:00<?, ?it/s]


✅ Sync Complete. Total: <MagicMock name='mock.aggregate.over_all().total_count' id='1935237513776'> objects.
Verified: tqdm called 3 times.
Success: Nested bar structure confirmed with matching arguments.


In [ ]:
import time
from tqdm.auto import tqdm

def simulate_ingestion(num_files=3, chunks_per_file=5, batch_size=2):
    print("🚀 Starting Simulated MathBrain Sync...\n")
    
    # Outer bar (Files)
    file_pbar = tqdm(range(num_files), desc="Files", position=0)
    
    for i in file_pbar:
        fname = f"document_{i+1}.tex"
        file_pbar.set_postfix({"current": fname})
        
        # Inner bar (Chunks) - positioned at 1 to sit below the main bar
        # leave=False keeps the UI clean after each file finishes
        chunk_pbar = tqdm(range(chunks_per_file), 
                          desc=f"  └ {fname}", 
                          position=1, 
                          leave=False)
        
        for j in chunk_pbar:
            # Simulate the "Add to Batch" step (instant)
            time.sleep(0.1) 
            
            # Simulate the "Batch Flush / Embedding" step (slow)
            if (j + 1) % batch_size == 0:
                chunk_pbar.set_postfix_str("Embedding...")
                time.sleep(0.8) # Mimic Ollama latency
                chunk_pbar.set_postfix_str("Done")
            
            chunk_pbar.update(1)
        
        chunk_pbar.close() # Clean up the inner bar
        
    print("\n✅ Simulation Complete!")

# Run the simulation
simulate_ingestion()

🚀 Starting Simulated MathBrain Sync...



Files:   0%|          | 0/3 [00:00<?, ?it/s]

  └ document_1.tex:   0%|          | 0/5 [00:00<?, ?it/s]

  └ document_2.tex:   0%|          | 0/5 [00:00<?, ?it/s]

  └ document_3.tex:   0%|          | 0/5 [00:00<?, ?it/s]


✅ Simulation Complete!


In [ ]:
# PathType = Union[str, os.PathLike]
# FileProcessor = Callable[[PathType], str]

# class MathBrainClient:
#     def __init__(
#         self, 
#         host: str = "localhost", 
#         port: int = 8080,
#         chunk_size: int = 800,
#         overlap: int = 150,
#         batch_size: int = 1
#     ) -> None:
#         self.client = weaviate.connect_to_local(host=host, port=port)
#         self.CHUNK_SIZE = chunk_size
#         self.OVERLAP = overlap
#         self.BATCH_SIZE = batch_size

#     def setup_collection(self, collection_name: str, force_recycle: bool = False) -> None:
#         exists = self.client.collections.exists(collection_name)
#         if force_recycle and exists:
#             print(f"Force Recycle: Deleting existing collection {collection_name}...")
#             self.client.collections.delete(collection_name)
#             exists = False

#         if not exists:
#             self.client.collections.create(
#                 name=collection_name,
#                 vector_config=Configure.Vectors.text2vec_ollama(
#                     name="default",
#                     api_endpoint="http://ollama:11434",
#                     model="nomic-embed-text",
#                     vector_index_config=Configure.VectorIndex.hnsw(
#                         distance_metric=VectorDistances.COSINE
#                     ),
#                 ),
#                 properties=[
#                     Property(name="content", data_type=DataType.TEXT),
#                     Property(name="fileName", data_type=DataType.TEXT),
#                     Property(name="filePath", data_type=DataType.TEXT),
#                     Property(name="contentHash", data_type=DataType.TEXT), # Added property
#                 ]
#             )
#             print(f"Collection `{collection_name}` initialized.")

#     def _split_text(self, text: str) -> List[str]:
#         text = re.sub(r'\n{3,}', '\n\n', text)
#         paragraphs = text.split('\n\n')
#         chunks: List[str] = []
#         current_chunk = ""

#         for para in paragraphs:
#             para = para.strip()
#             if not para: continue

#             if len(para) > self.CHUNK_SIZE:
#                 if current_chunk:
#                     chunks.append(current_chunk.strip())
#                     current_chunk = ""
#                 step = self.CHUNK_SIZE - self.OVERLAP
#                 for i in range(0, len(para), step):
#                     chunks.append(para[i : i + self.CHUNK_SIZE])
#                 continue

#             if len(current_chunk) + len(para) <= self.CHUNK_SIZE:
#                 current_chunk += (para + "\n\n")
#             else:
#                 if current_chunk:
#                     chunks.append(current_chunk.strip())
#                 overlap_text = current_chunk[-self.OVERLAP:] if len(current_chunk) > self.OVERLAP else ""
#                 current_chunk = overlap_text + para + "\n\n"

#         if current_chunk:
#             chunks.append(current_chunk.strip())
#         return [c[:self.CHUNK_SIZE].strip() for c in chunks if len(c) > 10]

#     def _get_file_hash(self, text: str) -> str:
#         return hashlib.md5(text.encode('utf-8')).hexdigest()

#     def ingest_files(
#         self, 
#         input_source: Union[PathType, Iterable[PathType]], 
#         collection_name: str = "MathDocument", 
#         force_recycle: bool = False, 
#         processor: Optional[FileProcessor] = None,
#         exclude_patterns: Optional[List[str]] = None
#     ) -> None:
#         self.setup_collection(collection_name, force_recycle)
#         collection = self.client.collections.get(collection_name)
        
#         # Setup Processor
#         def default_proc(p):
#             with open(os.fspath(p), "r", encoding="utf-8") as f: return f.read()
#         active_processor = processor or default_proc
#         ignores = exclude_patterns or []

#         # 1. Gather Files
#         all_paths = []
#         if isinstance(input_source, (str, os.PathLike)) and os.path.isdir(input_source):
#             for root, _, files in os.walk(input_source):
#                 for f in files:
#                     full_p = os.path.join(root, f)
#                     if f.lower().endswith((".tex", ".md", ".txt")):
#                         if not any(fnmatch.fnmatch(f, pat) or fnmatch.fnmatch(full_p, pat) for pat in ignores):
#                             all_paths.append(full_p)
#         else:
#             all_paths = [os.fspath(p) for p in input_source]

#         print(f"Syncing {len(all_paths)} files...")

#         pbar = tqdm(all_paths, desc="MathBrain Sync")
#         with collection.batch.fixed_size(batch_size=self.BATCH_SIZE) as batch:
#             for path in pbar:
#                 file_name = os.path.basename(path)
#                 pbar.set_postfix({"file": file_name[:20]})
                
#                 try:
#                     text = active_processor(path)
#                     if not text.strip(): continue
#                     current_hash = self._get_file_hash(text)

#                     # 2. SMART SKIP: Check if file + hash already exists
#                     existing = collection.query.fetch_objects(
#                         filters=(
#                             Filter.by_property("filePath").equal(str(path)) & 
#                             Filter.by_property("contentHash").equal(current_hash)
#                         ),
#                         limit=1,
#                         return_properties=[]
#                     )
                    
#                     if len(existing.objects) > 0 and not force_recycle:
#                         continue # File is unchanged, skip it!

#                     # 3. CLEANUP: Delete old chunks for this file
#                     collection.data.delete_many(
#                         where=Filter.by_property("filePath").equal(str(path))
#                     )
                    
#                     # 4. INDEX: Add new chunks
#                     chunks = self._split_text(text)
#                     for i, chunk in enumerate(chunks):
#                         batch.add_object(
#                             properties={
#                                 "content": chunk,
#                                 "fileName": file_name,
#                                 "filePath": str(path),
#                                 "contentHash": current_hash 
#                             },
#                             uuid=generate_uuid5(f"{path}_{i}")
#                         )
#                 except Exception as e:
#                     print(f"\n[Error] {file_name}: {e}")

#         final_count = collection.aggregate.over_all(total_count=True).total_count
#         print(f"\nSync Complete. Brain contains {final_count} objects.")

#     def close(self): self.client.close()
#     def __enter__(self): return self
#     def __exit__(self, *args): self.close()

#     def delete_collection(self, collection_name: str):
#         """Permanent deletion of a collection and all its vectors."""
#         if self.client.collections.exists(collection_name):
#             self.client.collections.delete(collection_name)
#             print(f"Collection '{collection_name}' has been deleted.")
#         else:
#             print(f"Deletion skipped: '{collection_name}' does not exist.")

In [ ]:
# # import random
# if __name__ == "__main__":
#     from pathlib import Path
    
#     # Example using pathlib.Path
#     DOC_DIR = Path(r"C:\Users\hyunj\Documents\Obsidian\Chores\math\_writing")

#     # samples = [
#     #     Path(r"C:\Users\hyunj\Documents\Obsidian\Chores\math\_writing\_definitions\definition_scheme.tex"),
#     #     Path(r"C:\Users\hyunj\Documents\Obsidian\Chores\math\_writing\_definitions\definition_categories_of_presheaves_and_sheaves_on_a_topological_space_valued_in_a_category.tex"),
#     #     Path(r"C:\Users\hyunj\Documents\Obsidian\Chores\math\_writing\_definitions\definition_sheaf_on_a_site.tex"),
#     #     ]

#     # 1. Gather all potential paths first
#     all_eligible_files = []
#     for root, _, files in os.walk(DOC_DIR):
#         for f in files:
#             if f.lower().endswith((".tex", ".md", ".txt")) and f != "main.tex":
#                 all_eligible_files.append(Path(root) / f)

#     # 2. Calculate 1% (minimum of 1 file so it doesn't fail on small dirs)
#     sample_size = max(1, int(len(all_eligible_files) * 0.01))
    
#     # 3. Randomly sample the list
#     random.seed(42)
#     sampled_paths = random.sample(all_eligible_files, sample_size)
    
#     print(f"Total files found: {len(all_eligible_files)}")
#     print(f"Sampling 1% -> {len(sampled_paths)} files for this test run.")


    

#     # You can now tune these parameters here!
#     with MathBrainClient(
#         chunk_size=2000, 
#         overlap=400, 
#         batch_size=10
#     ) as brain:
#         brain.ingest_files(
#             input_source=sampled_paths,
#             # DOC_DIR, 
#             collection_name="math_writing",
#             force_recycle=True, 
#             exclude_patterns=["main.tex"],
#             processor=latex_comment_stripping_processor,
#         )

In [ ]:
# with MathBrainClient() as brain:
#     brain.delete_collection("MathDocument")

In [ ]:
#| export
# import weaviate
# import weaviate.classes.query as wvc
# import weaviate
# from weaviate.classes.query import MetadataQuery

# class MathBrainSearcher:
#     def __init__(
#             self,
#             collection_name: str,
#             host: str = "localhost",
#             port: int = 8080
#             ) -> None:
#         self.client = weaviate.connect_to_local(host=host, port=port)
#         self.collection_name = collection_name
#         self.collection = self.client.collections.get(self.collection_name)
        
#         # Check if we are ready immediately
#         count = self.collection.aggregate.over_all(total_count=True).total_count
#         if count == 0:
#             print("Wait... Brain reports 0 objects. Checking again in 2 seconds...")
#             import time
#             time.sleep(2)

#     def search(self, query_text, limit=3):
#         """Perform a semantic/vector search."""
#         print(f"\nSearching for: '{query_text}'...")
        
#         response = self.collection.query.near_text(
#             query=query_text,
#             limit=limit,
#             return_metadata=wvc.MetadataQuery(distance=True)
#         )

#         if not response.objects:
#             print("No matches found. Is the brain empty?")
#             return

#         for i, obj in enumerate(response.objects):
#             print(f"\n--- Result #{i+1} (Distance: {obj.metadata.distance:.4f}) ---")
#             print(f"File: {obj.properties['fileName']}")
#             print(f"Path: {obj.properties['filePath']}")
#             print("-" * 30)
#             # Print first 500 chars of the content
#             content = obj.properties['content']
#             preview = (content[:500] + '...') if len(content) > 500 else content
#             print(preview)

#     def close(self):
#         self.client.close()

In [ ]:
#| export
import weaviate
import weaviate.classes.query as wvc
import time

class MathBrainSearcher:
    def __init__(self, collection_name: str, host: str = "localhost", port: int = 8080):
        self.client = weaviate.connect_to_local(host=host, port=port)
        self.collection = self.client.collections.get(collection_name)

    def search(self, query: str, alpha: float = 0.5, limit: int = 3, top_k: int = 20, rerank: bool = False):
        """Unified search with score and vector distance tracking."""
        start_time = time.time()
        fetch_count = max(top_k, limit) if rerank else limit

        response = self.collection.query.hybrid(
            query=query,
            alpha=alpha,
            limit=fetch_count,
            rerank=wvc.Rerank(prop="content", query=query) if rerank else None,
            # We now request both Score (Blended) and Distance (Vector Only)
            return_metadata=wvc.MetadataQuery(score=True, distance=True)
        )

        results = response.objects[:limit]
        self._display_summary(len(results), time.time() - start_time)
        self._display_results(results)

    def _display_summary(self, count: int, duration: float):
        print(f"\nFound {count} matches in {duration:.3f} seconds.")
        print("=" * 60)

    def _display_results(self, objects):
        if not objects:
            print("No matches found.")
            return
        for i, obj in enumerate(objects):
            self._print_single_object(i + 1, obj)

    def _print_single_object(self, rank: int, obj):
        props = obj.properties
        # Score: Higher is better | Distance: Lower is better
        score = obj.metadata.score or 0.0
        dist = f"{obj.metadata.distance:.4f}" if obj.metadata.distance is not None else "N/A (Keyword match)"
        
        print(f"Result #{rank}")
        print(f" > Hybrid Score: {score:.4f} (Higher is better)")
        print(f" > Vector Dist:  {dist} (Lower is better)")
        print(f"File: {props.get('fileName')}")
        print("-" * 30)
        
        content = props.get('content', "")
        preview = (content[:500] + '...') if len(content) > 500 else content
        print(f"{preview}\n")

    def close(self):
        self.client.close()

    def __enter__(self): return self
    def __exit__(self, *args): self.close()

# import weaviate
# import weaviate.classes.query as wvc
# import time

# class MathBrainSearcher:
#     def __init__(self, collection_name: str, host: str = "localhost", port: int = 8080):
#         self.client = weaviate.connect_to_local(host=host, port=port)
#         self.collection = self.client.collections.get(collection_name)

#     def search(self, query: str, alpha: float = 0.5, limit: int = 3, top_k: int = 20, rerank: bool = False):
#         """Unified search with score and vector distance tracking."""
#         start_time = time.time()
#         fetch_count = max(top_k, limit) if rerank else limit

#         response = self.collection.query.hybrid(
#             query=query,
#             alpha=alpha,
#             limit=fetch_count,
#             rerank=wvc.Rerank(prop="content", query=query) if rerank else None,
#             # We now request both Score (Blended) and Distance (Vector Only)
#             return_metadata=wvc.MetadataQuery(score=True, distance=True)
#         )

#         results = response.objects[:limit]
#         self._display_summary(len(results), time.time() - start_time)
#         self._display_results(results)

#     def _display_summary(self, count: int, duration: float):
#         print(f"\nFound {count} matches in {duration:.3f} seconds.")
#         print("=" * 60)

#     def _display_results(self, objects):
#         if not objects:
#             print("No matches found.")
#             return
#         for i, obj in enumerate(objects):
#             self._print_single_object(i + 1, obj)

#     def _print_single_object(self, rank: int, obj):
#         props = obj.properties
#         # Score: Higher is better | Distance: Lower is better
#         score = obj.metadata.score or 0.0
#         dist = f"{obj.metadata.distance:.4f}" if obj.metadata.distance is not None else "N/A (Keyword match)"
        
#         print(f"Result #{rank}")
#         print(f" > Hybrid Score: {score:.4f} (Higher is better)")
#         print(f" > Vector Dist:  {dist} (Lower is better)")
#         print(f"File: {props.get('fileName')}")
#         print("-" * 30)
        
#         content = props.get('content', "")
#         preview = (content[:500] + '...') if len(content) > 500 else content
#         print(f"{preview}\n")

#     def close(self):
#         self.client.close()

#     def __enter__(self): return self
#     def __exit__(self, *args): self.close()

In [ ]:
# #| notest
# collection_name = 'math_writing'
# if __name__ == "__main__":
#     searcher = MathBrainSearcher(collection_name=collection_name)
    
#     try:
#         while True:
#             user_query = input("\nAsk your Math Brain a question (or 'q' to quit): ")
#             if user_query.lower() == 'q':
#                 break
            
#             searcher.search(user_query, limit=10)
#     finally:
#         searcher.close()